# 🤖 Workshop Day 2: Building Agents — สร้าง Agent ด้วย Google ADK

**Agentic RAG Workshop** | Day 2 of 3

---

### 🎯 วันนี้เราจะเรียนรู้
1. **Agent คืออะไร** — ต่างจาก Chatbot ยังไง
2. **Google ADK** — เครื่องมือสร้าง Agent จาก Google
3. **Tool** — สอน Agent ให้ "ทำงาน" ได้
4. **RAG Tool** — ต่อ Agent เข้ากับ VectorDB ที่สร้างจาก Day 1
5. **Multi-Agent** — หลาย Agent ทำงานร่วมกัน
6. **Agentic RAG** เต็มรูปแบบ — ค้นหา + สรุป + จำบริบท

### 🗺️ ภาพรวม Pipeline

| Day 1 (Data Engineering) | ➡️ | Day 2 (Agent Building) |
|---|---|---|
| Raw Data → Dedup → Chunk | | 2.1 Agent แรก |
| → Embed | | 2.2 เพิ่ม Tool |
| → Index (Qdrant) | ➡️ | 2.3 RAG Tool + Qdrant ⭐ |
| → Retrieve | | 2.4 Multi-Agent |
| | | 2.5 Session & Memory |
| | | 2.6 Agentic RAG เต็ม ⭐ |

## 📦 Section 0: ติดตั้ง Dependencies

> ⚠️ Google ADK ปกติใช้บน terminal ด้วยคำสั่ง `adk web` แต่เราจะปรับให้ทำงานบน Colab ได้

### 🆚 ADK vs LangChain vs CrewAI

| | **Google ADK** | **LangChain** | **CrewAI** |
|---|:---:|:---:|:---:|
| **เหมาะกับ** | Multi-Agent + Gemini | เชน LLM+Tool | ทีม Agent |
| **LLM** | Gemini (หลัก) + อื่นๆ | ทุก LLM | ทุก LLM |
| **Multi-Agent** | ✅ Built-in | ❌ ต้องเขียนเอง | ✅ Built-in |
| **Workflow** | Sequential/Parallel/Loop | LCEL Chain | Sequential/Hierarchical |
| **Dev UI** | ✅ `adk web` | ❌ | ❌ |
| **พัฒนาโดย** | Google | LangChain Inc. | CrewAI Inc. |

> 💡 เราเลือก **ADK** เพราะเป็น framework ล่าสุดจาก Google ออกแบบมาสำหรับ Multi-Agent โดยเฉพาะ
> และทำงานร่วมกับ **Gemini** ได้ดีที่สุด

In [ ]:
%%time
# ติดตั้ง Google ADK และ libraries จาก Day 1
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-adk', 'google-genai', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters']:
    _pip_install(_pkg)

print('✅ ติดตั้งเรียบร้อย!')

In [ ]:
%%time
# ตั้งค่า Gemini API Key
import os

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ โหลด API Key จาก Colab Secrets')
except Exception:
    api_key = input('🔑 กรุณาวาง Gemini API Key: ')
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

# ทดสอบ API
from google import genai
try:
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    resp = client.models.generate_content(model='gemini-2.5-flash', contents='ตอบแค่ สวัสดี')
    print(f'✅ API ทำงาน: {(resp.text.strip() if resp.text else '(ไม่มี output)')}')
except Exception as e:
    print(f'❌ API Error: {e}')
    print('💡 ตรวจสอบ: 1) API Key ถูกต้อง? 2) มี internet? 3) Free tier หมดโควต้า?')

### 📚 รู้จัก Gemini API Parameters

เมื่อเรียก Gemini (ทั้งโดยตรงหรือผ่าน ADK) สามารถปรับ parameter ได้หลายตัว:

#### 🎛️ Generation Parameters

| Parameter | ค่า Default | ช่วง | ทำหน้าที่ |
|-----------|:-----------:|------|----------|
| `temperature` | 1.0 | 0.0 - 2.0 | ความ "สร้างสรรค์" ของคำตอบ |
| `top_p` | 0.95 | 0.0 - 1.0 | จำกัดกลุ่มคำที่จะเลือก (nucleus sampling) |
| `top_k` | 40 | 1 - ∞ | จำนวนคำที่พิจารณาในแต่ละ step |
| `max_output_tokens` | 8192 | 1 - 65536 | จำกัดความยาวคำตอบ |

#### 🌡️ Temperature — ปรับความสร้างสรรค์

```
temperature = 0.0  →  ตอบเหมือนเดิมทุกครั้ง (deterministic)
                       เหมาะกับ: ตรวจงาน, สรุปข้อเท็จจริง, RAG

temperature = 1.0  →  สมดุลระหว่างสร้างสรรค์และถูกต้อง
                       เหมาะกับ: chatbot ทั่วไป, Agent

temperature = 2.0  →  สร้างสรรค์สูง อาจ "หลุด" บ้าง
                       เหมาะกับ: brainstorm, แต่งเรื่อง
```

#### 🔍 top_p vs top_k

| | top_k | top_p |
|---|------|------|
| **วิธีเลือกคำ** | เลือกจาก k คำที่น่าจะเป็นที่สุด | เลือกจากคำที่รวม probability ≤ p |
| **ตัวอย่าง** | top_k=3 → เลือกจาก 3 คำ | top_p=0.9 → เลือกจากคำที่รวมได้ 90% |
| **ปรับแล้วได้** | ยิ่งน้อย → ยิ่ง conservative | ยิ่งน้อย → ยิ่ง conservative |

#### 🏷️ Gemini Models ที่ใช้ได้

| Model | ความเร็ว | ราคา | เหมาะกับ |
|-------|:--------:|:----:|----------|
| `gemini-2.5-flash` | ⚡ เร็ว | 💰 ถูก | Agent, Tool calling, RAG |
| `gemini-2.5-pro` | 🐢 ช้ากว่า | 💰💰 แพง | วิเคราะห์เชิงลึก, ตรวจงาน |
| `gemini-3.1-flash` | ⚡⚡ เร็วสุด | 💰 ถูก | Agent รุ่นใหม่, งานทั่วไป |

> 💡 ใน Workshop เราใช้ **`gemini-2.5-flash`** เพราะเร็ว ถูก และรองรับ Tool calling ดี

In [ ]:
%%time
# ─── ทดลองปรับ Temperature ───
from google import genai

prompt = 'แต่งประโยคเกี่ยวกับ AI 1 ประโยค'

print('🧪 ทดสอบ Temperature (ถามซ้ำ 3 รอบ)\n')

for temp in [0.0, 1.0, 2.0]:
    print(f'--- temperature = {temp} ---')
    for round_num in range(1, 4):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=genai.types.GenerateContentConfig(
                    temperature=temp,
                    max_output_tokens=100,
                )
            )
            print(f'  รอบ {round_num}: {(resp.text.strip() if resp.text else '(ไม่มี output)')[:80]}')
        except Exception as e:
            print(f'  ❌ Error: {e}')
    print()

print('💡 สังเกต: temp=0.0 → ตอบเหมือนกันทุกรอบ, temp=2.0 → ตอบต่างกัน')

In [ ]:
%%time
# ─── ทดลองปรับ max_output_tokens ───
print('🧪 ทดสอบ max_output_tokens\n')

for max_tokens in [20, 100, 500]:
    try:
        resp = client.models.generate_content(
            model='gemini-2.5-flash',
            contents='อธิบาย AI คืออะไร',
            config=genai.types.GenerateContentConfig(
                max_output_tokens=max_tokens,
                temperature=0.5
            )
        )
        text = resp.text.strip() if resp.text else '(ไม่มี output — token น้อยเกินไป)'
        print(f'max_tokens={max_tokens:>3}: ({len(text)} chars) {text[:80]}...')
    except Exception as e:
        print(f'  ❌ Error: {e}')

print('\n💡 max_output_tokens จำกัด "ความยาว" ไม่ใช่ "ความถูกต้อง"')

### 💡 สังเกต
- `temperature=0.0` → ตอบ **เหมือนกัน** ทุกรอบ — เหมาะกับ RAG ที่ต้องการคำตอบคงที่
- `temperature=2.0` → ตอบ **ต่างกัน** ทุกรอบ — เหมาะกับงาน creative
- `max_output_tokens` น้อย → คำตอบถูกตัด ไม่ใช่ "สรุปให้สั้นลง"

> 🎯 สำหรับ **Agent/RAG** แนะนำ: `temperature=0.5-1.0`, `max_output_tokens=2048+`

---
## 🤖 Section 2.1: Agent แรกของคุณ

### Agent vs Chatbot — ต่างกันยังไง?

| | Chatbot | Agent |
|---|---------|-------|
| **วิธีทำงาน** | ตอบคำถามอย่างเดียว | ตัดสินใจ + เรียกใช้เครื่องมือ |
| **ความสามารถ** | ตอบจากสิ่งที่รู้ | ค้นหาข้อมูล, คำนวณ, เรียก API |
| **ความจำ** | ส่วนใหญ่ไม่จำ | จำบริบทได้ |
| **ตัวอย่าง** | FAQ Bot | ChatGPT + Plugins, Gemini |

### Google ADK คืออะไร?

**Agent Development Kit (ADK)** = open-source Python framework จาก Google

```
Agent = Model (LLM) + Instructions (วิธีคิด) + Tools (เครื่องมือ)
```

- ใช้ **Gemini** เป็น LLM หลัก (รองรับ model อื่นด้วย)
- สร้าง **Multi-Agent** ได้ง่าย
- มี **Web UI** สำหรับ debug

In [ ]:
%%time
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types

import warnings
warnings.filterwarnings('ignore')

# ─── Agent แรก: ไม่มี Tool มีแค่ Instructions ───
greeting_agent = Agent(
    name='greeting_agent',
    model='gemini-2.5-flash',
    description='Agent ที่ทักทายและตอบคำถามเป็นภาษาไทย',
    instruction="""คุณเป็น AI Assistant ภาษาไทย
    - ทักทายอย่างสุภาพ
    - ตอบสั้นกระชับ ไม่เกิน 2 ประโยค
    - ใช้ emoji เหมาะสม
    """
)

print('✅ สร้าง Agent แรกสำเร็จ!')
print(f'   ชื่อ: {greeting_agent.name}')
print(f'   Model: {greeting_agent.model}')

### 🔧 ทำไมต้องใช้ Async + Runner?

- **`async/await`** — ADK ออกแบบให้ทำงานแบบ asynchronous เพราะ Agent รอ LLM ตอบ (I/O bound)
- **`InMemoryRunner`** — ตัวรัน Agent ในหน่วยความจำ จัดการ event loop ให้
- **`InMemorySessionService`** — เก็บประวัติสนทนาใน RAM

```
Runner → สร้าง Session → ส่งข้อความให้ Agent → Agent ตัดสินใจ
  → เรียก Tool (ถ้าจำเป็น) → สร้างคำตอบ → ส่ง Event กลับ
```

> 💡 บน Colab ใช้ `await` ได้โดยตรงใน cell (ไม่ต้อง `asyncio.run()`)

In [ ]:
# ─── ฟังก์ชันช่วยคุย กับ Agent บน Colab ───
import asyncio

async def chat_with_agent(runner, message, user_id='user_1', session_id='default_session'):
    """ส่งข้อความถึง Agent แล้วรับคำตอบ"""
    content = types.Content(
        role='user',
        parts=[types.Part.from_text(text=message)]
    )
    
    final_response = ''
    tool_calls = []
    
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_response = part.text
                if part.function_call:
                    tool_calls.append({
                        'name': part.function_call.name,
                        'args': dict(part.function_call.args) if part.function_call.args else {}
                    })
    
    if tool_calls:
        print(f'  🔧 Tools เรียกใช้: {[t["name"] for t in tool_calls]}')
    
    return final_response


async def demo_chat(agent, messages):
    """Demo สนทนาหลายรอบกับ Agent (ใช้ session เดียวกันตลอด)"""
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session_id = f'session_{id(agent)}'
    
    # สร้าง session ผ่าน runner
    await runner.session_service.create_session(
        app_name=agent.name,
        user_id='user_1',
        session_id=session_id
    )
    
    for msg in messages:
        print(f'\n👤 ผู้ใช้: {msg}')
        response = await chat_with_agent(runner, msg, session_id=session_id)
        print(f'🤖 Agent: {response}')


print('✅ ฟังก์ชัน chat พร้อมใช้งาน')

In [ ]:
# ─── ทดสอบ Agent แรก ───
await demo_chat(greeting_agent, [
    'สวัสดีครับ',
    'AI คืออะไรอธิบายสั้นๆ',
    'ขอบคุณครับ'
])

### 💡 สังเกต
- Agent ตอบ **ตาม instruction** ที่เรากำหนด (สุภาพ, สั้น, มี emoji)
- ถ้าเปลี่ยน instruction → บุคลิก Agent เปลี่ยนทันที
- Agent ยังไม่มี Tool → ตอบได้แค่จาก **ความรู้ของ LLM** เท่านั้น

> 🎯 **Instruction คือจิตวิญญาณของ Agent** — เขียนดี = Agent ฉลาด

### 🧪 แบบฝึกหัด 2.1

ลองเปลี่ยน `instruction` ของ Agent ให้มีบุคลิกต่างกัน:
- เป็นอาจารย์สอน AI
- เป็นหมอให้คำปรึกษาสุขภาพ
- เป็นที่ปรึกษาการเงิน

In [ ]:
# 🧪 แบบฝึกหัด: เปลี่ยน instruction ให้ Agent มีบุคลิกใหม่
# my_agent = Agent(
#     name='...',
#     model='gemini-2.5-flash',
#     description='...',
#     instruction='...'
# )
# await demo_chat(my_agent, ['สวัสดี', 'ช่วยแนะนำหน่อย'])

### ✍️ Tips: เขียน Instruction ที่ดี

**Instruction = สมองของ Agent** — เขียนดีได้ผลดี เขียนกว้างได้ผลไม่แน่นอน

| เทคนิค | ตัวอย่าง |
|--------|--------|
| **ระบุ Role** | "คุณเป็นผู้เชี่ยวชาญด้านสุขภาพ" |
| **บอกเมื่อไหร่ใช้ Tool** | "ใช้ calculate_bmi เมื่อถูกถามเรื่องน้ำหนัก" |
| **บอกเมื่อไหร่ตอบเอง** | "คำถามทั่วไป เช่น ทักทาย → ตอบเอง" |
| **กำหนด Format** | "ตอบเป็น bullet points ไม่เกิน 5 ข้อ" |
| **ใส่ Constraint** | "ห้ามเดา ถ้าไม่รู้ให้บอกตรงๆ" |
| **ภาษา** | "ตอบเป็นภาษาไทย ใช้ emoji" |

```python
# ❌ Instruction แย่
instruction = "ตอบคำถาม"

# ✅ Instruction ดี
instruction = """คุณเป็นผู้เชี่ยวชาญด้านสุขภาพ ตอบเป็นภาษาไทย
- ใช้ calculate_bmi เมื่อถูกถามเรื่อง BMI หรือน้ำหนัก
- คำถามทั่วไป → ตอบเอง ไม่ต้องเรียก tool
- ตอบกระชับ ไม่เกิน 3 ประโยค
- ห้ามเดา ถ้าไม่แน่ใจให้แนะนำพบแพทย์"""
```

---
## 🔧 Section 2.2: สร้าง Tool ให้ Agent

### Tool คืออะไร?

**Tool = Python function ที่ Agent เรียกใช้ได้**

```
ผู้ใช้ถาม "BMI ของฉันเท่าไหร่ น้ำหนัก 70 kg สูง 175 cm"
    ↓
Agent อ่านคำถาม → ตัดสินใจ: ต้องใช้ tool calculate_bmi!
    ↓
Agent เรียก: calculate_bmi(weight_kg=70, height_cm=175)
    ↓
Agent ได้ผลลัพธ์ → สร้างคำตอบให้ผู้ใช้
```

### 📌 กฎสำคัญในการเขียน Tool

| กฎ | ทำไม? |
|-----|-------|
| ต้องมี **docstring** | LLM อ่าน docstring เพื่อเลือก tool |
| ต้องมี **type hints** | LLM รู้ว่าต้องส่ง parameter อะไร |
| return **dict** | Agent อ่านผลลัพธ์จาก dict |
| ชื่อฟังก์ชันชัดเจน | LLM เลือก tool จากชื่อด้วย |

In [ ]:
%%time
# ─── Tool 1: คำนวณ BMI ───
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """คำนวณค่าดัชนีมวลกาย (BMI) จากน้ำหนักและส่วนสูง

    Args:
        weight_kg: น้ำหนักเป็นกิโลกรัม
        height_cm: ส่วนสูงเป็นเซนติเมตร

    Returns:
        dict: ค่า BMI, ระดับน้ำหนัก, และคำแนะนำ
    """
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)

    if bmi < 18.5:
        level = 'น้ำหนักน้อยเกินไป'
        advice = 'ควรเพิ่มน้ำหนักด้วยอาหารที่มีโปรตีนสูง'
    elif bmi < 25:
        level = 'น้ำหนักปกติ'
        advice = 'รักษาน้ำหนักนี้ไว้ ออกกำลังกายสม่ำเสมอ'
    elif bmi < 30:
        level = 'น้ำหนักเกิน'
        advice = 'ควรลดน้ำหนักด้วยการควบคุมอาหารและออกกำลังกาย'
    else:
        level = 'อ้วน'
        advice = 'ควรปรึกษาแพทย์เพื่อวางแผนลดน้ำหนัก'

    return {'bmi': round(bmi, 1), 'level': level, 'advice': advice}


# ─── Tool 2: แปลงอุณหภูมิ ───
def convert_temperature(value: float, from_unit: str) -> dict:
    """แปลงอุณหภูมิระหว่าง Celsius และ Fahrenheit

    Args:
        value: ค่าอุณหภูมิ
        from_unit: หน่วยต้นทาง ('celsius' หรือ 'fahrenheit')

    Returns:
        dict: ค่าอุณหภูมิที่แปลงแล้ว
    """
    if from_unit.lower() == 'celsius':
        result = (value * 9/5) + 32
        return {'result': round(result, 1), 'from': f'{value}°C', 'to': f'{result:.1f}°F'}
    elif from_unit.lower() == 'fahrenheit':
        result = (value - 32) * 5/9
        return {'result': round(result, 1), 'from': f'{value}°F', 'to': f'{result:.1f}°C'}
    else:
        return {'error': f'ไม่รู้จักหน่วย {from_unit}'}


# ─── สร้าง Agent พร้อม Tools ───
health_agent = Agent(
    name='health_agent',
    model='gemini-2.5-flash',
    description='Agent สุขภาพ คำนวณ BMI และแปลงอุณหภูมิ',
    instruction="""คุณเป็น Agent ด้านสุขภาพ ตอบเป็นภาษาไทย
    - ใช้ calculate_bmi เมื่อถูกถามเรื่อง BMI หรือน้ำหนัก
    - ใช้ convert_temperature เมื่อถูกถามเรื่องอุณหภูมิ
    - ถ้าคำถามไม่เกี่ยวกับ tools ให้ตอบจากความรู้ทั่วไป
    """,
    tools=[calculate_bmi, convert_temperature]
)

print('✅ สร้าง Health Agent พร้อม 2 tools')
print(f'   🔧 Tools: {[t.__name__ for t in health_agent.tools]}')

In [ ]:
runner = InMemoryRunner(agent=health_agent, app_name=health_agent.name)
await runner.session_service.create_session(app_name=health_agent.name, user_id='user_1', session_id='test_session')
# ─── ทดสอบ Tool Selection ───
print('═' * 60)
print('🧪 ทดสอบ: Agent เลือก Tool ไหน?')
print('═' * 60)

test_messages = [
    'น้ำหนัก 70 kg สูง 175 cm BMI เท่าไหร่',
    '37 องศาเซลเซียส เท่ากับกี่ฟาเรนไฮต์',
    'AI คืออะไร',  # ไม่ต้องใช้ tool!
]

for msg in test_messages:
    print(f'\n👤 ผู้ใช้: {msg}')
    response = await chat_with_agent(runner, msg, session_id='test_session')
    print(f'🤖 Agent: {response}')

### 💡 สังเกต
- คำถาม BMI → Agent เรียก `calculate_bmi` 🔧
- คำถามอุณหภูมิ → Agent เรียก `convert_temperature` 🔧
- คำถามทั่วไป → Agent **ไม่เรียก tool** ตอบเอง 💬

> Agent **ตัดสินใจเอง** ว่าจะใช้ tool ไหน — นี่คือจุดต่างจาก chatbot!

### 🧪 แบบฝึกหัด 2.2

สร้าง tool ของตัวเอง 1 ตัว แล้วเพิ่มเข้า Agent:
- คำนวณเกรดจากคะแนน (score → grade)
- คำนวณดอกเบี้ย
- แปลงสกุลเงิน
- อะไรก็ได้ที่คิดออก!

In [ ]:
# 🧪 แบบฝึกหัด: สร้าง tool ของตัวเอง
# def my_tool(param1: float, param2: str) -> dict:
#     """อธิบาย tool (สำคัญ! LLM อ่านอันนี้)"""
#     ...
#     return {...}
#
# my_agent = Agent(
#     name='my_agent', model='gemini-2.5-flash',
#     tools=[my_tool, calculate_bmi]
# )
# await demo_chat(my_agent, ['ทดสอบ tool ใหม่'])

---
## 🔍 Section 2.3: RAG Tool — ต่อ Agent เข้ากับ Qdrant ⭐

### Agentic RAG คืออะไร?

```
RAG ธรรมดา:     ถาม → ค้นหา → ตอบ              (ค้นทุกครั้ง ไม่ว่าจะถามอะไร)
Agentic RAG:    ถาม → Agent ตัดสินใจ → ค้นหา? → ตอบ    (ค้นเมื่อจำเป็น)
```

### ทำไม Agentic RAG ดีกว่า?

| | RAG ธรรมดา | Agentic RAG |
|---|----------|-------------|
| ถาม "สวัสดี" | ค้น VectorDB (เสียเวลา) | ตอบเลย ไม่ค้น ✅ |
| ถาม "AI ช่วยธนาคาร" | ค้น VectorDB ✅ | ค้น VectorDB ✅ |
| ถาม follow-up | ค้นใหม่ทุกครั้ง | จำบริบท + ค้นเมื่อต้อง ✅ |
| ใช้หลาย tool | ❌ ทำไม่ได้ | ✅ เลือก tool เอง |

### ตัวอย่าง Real-World
- **KADE (กสิกร)**: Agent ตัดสินใจว่าคำถามเรื่อง สินเชื่อ/บัตรเครดิต/ลงทุน แล้วค้นจาก KB ที่เกี่ยวข้อง
- **จุฬา Study Bot**: Agent ถูกถาม "สอบอะไรบ้าง" → ค้นตำรา, ถูกถาม "สวัสดี" → ทักทาย

> Agent ตัดสินใจเองว่า **เมื่อไหร่** ต้องค้นหาข้อมูล และ **ค้นจากที่ไหน**

In [ ]:
%%time
# ─── เตรียม Qdrant VectorDB (ข้อมูลเดียวกับ Day 1) ───
import hashlib, random
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

# โหลด Embedding Model
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print('✅ โหลด embedding model')

# ข้อมูลเดียวกับ Day 1
documents = {
    'healthcare': 'โรงพยาบาลศิริราช นำ AI มาช่วยวิเคราะห์ภาพ X-ray ทรวงอก ลดเวลาวินิจฉัยจาก 30 นาทีเหลือ 5 นาที ระบบ Deep Learning ตรวจจับความผิดปกติได้แม่นยำ 95% ช่วยแพทย์คัดกรองผู้ป่วยวัณโรคได้เร็วขึ้น ลดการรอคอยผลตรวจ และเพิ่มโอกาสรักษาได้ทันเวลา',
    'banking': 'ธนาคารกสิกรไทย พัฒนา KADE AI Assistant ใช้ RAG ดึงข้อมูลผลิตภัณฑ์ทางการเงินมาตอบลูกค้า ลดเวลาตอบคำถามจาก 5 นาทีเหลือ 30 วินาที ระบบเข้าใจภาษาไทยและอังกฤษ จัดการคำถามเรื่องสินเชื่อ บัตรเครดิต และการลงทุนได้อัตโนมัติ',
    'education': 'มหาวิทยาลัย จุฬาลงกรณ์ สร้างระบบ RAG ถาม-ตอบจากตำราเรียน แปลง PDF มากกว่า 500 เล่มเป็น vector embeddings ใช้ multilingual model รองรับภาษาไทย นักศึกษาสามารถถามคำถามและได้คำตอบพร้อมอ้างอิงหน้าหนังสือ ช่วยลดเวลาค้นหาจากหลายชั่วโมงเหลือไม่กี่วินาที',
    'agriculture': 'กรมวิชาการเกษตร ใช้ AI วิเคราะห์ภาพถ่ายดาวเทียมและโดรน ตรวจจับโรคพืชในไร่ข้าว พัฒนาระบบ Computer Vision ตรวจหาโรคไหม้ โรคใบจุด ด้วยความแม่นยำ 92% ช่วยเกษตรกรรู้ปัญหาเร็วขึ้น ลดความเสียหายจากโรคพืชได้ 40%',
    'kmitl': 'สถาบันพระจอมเกล้าเจ้าคุณทหารลาดกระบัง (KMITL) คณะวิศวกรรมศาสตร์ พัฒนาระบบ AI สำหรับ Smart Campus ใช้ IoT sensor รวมกับ Machine Learning วิเคราะห์การใช้พลังงานในอาคาร ลดค่าไฟได้ 25% นอกจากนี้ยังพัฒนาระบบ NLP ภาษาไทยสำหรับวิเคราะห์ความคิดเห็นนักศึกษา และนำ RAG มาสร้างระบบ AI Tutor ช่วยตอบคำถามวิชาเรียนจากเอกสารประกอบการสอน',
    'nlp': 'การประมวลผลภาษาธรรมชาติ หรือ NLP สำหรับภาษาไทยมีความท้าทายเฉพาะ เนื่องจากภาษาไทยไม่มีเว้นวรรคระหว่างคำ ทำให้ต้องใช้ Word Segmentation เฉพาะทาง เช่น PyThaiNLP นอกจากนี้ สระ วรรณยุกต์ และตัวเลขไทย ยังเพิ่มความซับซ้อนในการ tokenization',
    'rag_architecture': 'สถาปัตยกรรม RAG ประกอบด้วย 3 ส่วนหลัก ได้แก่ Retriever ที่ค้นหาเอกสารจาก VectorDB, Reader ที่อ่านและเข้าใจเอกสาร, และ Generator ที่สร้างคำตอบจากข้อมูลที่พบ ระบบ RAG ช่วยลดปัญหา hallucination ของ LLM เพราะคำตอบมาจากข้อมูลจริง',
    'embedding': 'Text Embedding แปลงข้อความเป็น vector ตัวเลข โมเดล multilingual-e5-large ของ Microsoft รองรับมากกว่า 100 ภาษารวมถึงภาษาไทย สร้าง vector ขนาด 1024 มิติ ข้อความที่มีความหมายคล้ายกันจะมี vector ที่อยู่ใกล้กันในพื้นที่ embedding',
    'vector_db': 'Qdrant เป็น Vector Database ที่มีประสิทธิภาพสูง รองรับการค้นหาแบบ ANN (Approximate Nearest Neighbor) ทำให้ค้นหาได้เร็วแม้มีข้อมูลหลายล้าน vector รองรับทั้ง Cosine Similarity, Dot Product, และ Euclidean Distance',
}

# Chunk + Embed + Index
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

all_chunks = []
all_sources = []
for source, text in documents.items():
    chunks = splitter.split_text(text)
    for chunk in chunks:
        all_chunks.append(chunk)
        all_sources.append(source)

print(f'📊 {len(all_chunks)} chunks จาก {len(documents)} เอกสาร')

# Embed
passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]
print(f'✅ Embedding: {embeddings.shape}')

# Create Qdrant collection
qdrant = QdrantClient(':memory:')
qdrant.create_collection(
    collection_name='thai_ai_kb',
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

# Upsert
points = [
    models.PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={'text': all_chunks[i], 'source': all_sources[i]}
    )
    for i in range(len(all_chunks))
]
qdrant.upsert(collection_name='thai_ai_kb', points=points)
print(f'✅ Upsert {len(points)} points เข้า Qdrant')

In [ ]:
%%time
# ─── สร้าง RAG Tool ───
def search_thai_ai_knowledge(query: str) -> dict:
    """ค้นหาข้อมูลเกี่ยวกับ AI ในประเทศไทย จากฐานความรู้

    ใช้ tool นี้เมื่อผู้ใช้ถามเกี่ยวกับ:
    - การประยุกต์ใช้ AI ในไทย (การแพทย์ ธนาคาร การศึกษา เกษตร)
    - เทคนิค RAG, Embedding, Vector Database
    - NLP ภาษาไทย

    Args:
        query: คำถามที่ต้องการค้นหา เป็นภาษาไทยหรืออังกฤษ

    Returns:
        dict: ผลลัพธ์จากฐานความรู้
    """
    query_vector = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points(
        collection_name='thai_ai_kb',
        query=query_vector,
        limit=3
    ).points

    if not results:
        return {'status': 'no_results', 'message': 'ไม่พบข้อมูลที่เกี่ยวข้อง'}

    contexts = []
    for r in results:
        contexts.append({
            'text': r.payload['text'],
            'source': r.payload['source'],
            'score': round(r.score, 4)
        })

    return {
        'status': 'success',
        'num_results': len(contexts),
        'results': contexts
    }

print('✅ RAG Tool พร้อมใช้งาน')

# ทดสอบ tool ตรงๆ
test = search_thai_ai_knowledge('AI ช่วยหมอวินิจฉัยโรค')
print(f'\n🧪 ทดสอบ: {test["num_results"]} ผลลัพธ์')
for r in test['results']:
    print(f'  [{r["source"]}] score={r["score"]} | {r["text"][:60]}...')

In [ ]:
%%time
# ─── สร้าง RAG Agent ───
rag_agent = Agent(
    name='thai_ai_expert',
    model='gemini-2.5-flash',
    description='ผู้เชี่ยวชาญ AI ในประเทศไทย ค้นหาจากฐานความรู้',
    instruction="""คุณเป็นผู้เชี่ยวชาญด้าน AI ในประเทศไทย

    กฎ:
    1. เมื่อผู้ใช้ถามเรื่อง AI ในไทย, case study, หรือเทคนิค RAG
       → ใช้ search_thai_ai_knowledge ค้นหาก่อนตอบเสมอ
    2. ตอบโดยอ้างอิงข้อมูลจากฐานความรู้ ระบุแหล่งที่มา
    3. ถ้าไม่พบข้อมูล ให้บอกตรงๆ ว่า "ไม่มีข้อมูลในฐานความรู้"
    4. คำถามทั่วไป เช่น ทักทาย → ตอบเองโดยไม่ต้องค้น
    5. ตอบเป็นภาษาไทย กระชับ ใช้ bullet points
    """,
    tools=[search_thai_ai_knowledge]
)

print('✅ สร้าง RAG Agent สำเร็จ!')

In [ ]:
# ─── Demo: Agent ตัดสินใจเอง ───
print('═' * 60)
print('🧪 Demo: Agent เลือกเมื่อไหร่จะค้นหา เมื่อไหร่จะตอบเอง')
print('═' * 60)

await demo_chat(rag_agent, [
    'สวัสดี',                                  # → ตอบเอง
    'AI ช่วยวงการแพทย์ไทยอย่างไร',              # → ค้น Qdrant
    'RAG มีสถาปัตยกรรมอย่างไร',                 # → ค้น Qdrant
    'ขอบคุณครับ',                               # → ตอบเอง
])

### 💡 สังเกต
- "สวัสดี" → Agent **ไม่เรียก tool** ตอบเอง 💬
- "AI ช่วยวงการแพทย์" → Agent **เรียก** `search_thai_ai_knowledge` 🔧
- คำตอบ **อ้างอิงข้อมูลจริง** จาก Qdrant ไม่ใช่แต่งเอง

> นี่คือหัวใจ **Agentic RAG**: Agent ตัดสินใจเอง → ค้นเมื่อจำเป็น → ตอบจากข้อมูลจริง

### 🧪 แบบฝึกหัด 2.3

1. ถาม RAG Agent เกี่ยวกับ "NLP ภาษาไทย" และ "Vector Database" ดูว่าตอบจากฐานความรู้มั้ย
2. ลองเพิ่มข้อมูลใหม่เข้า Qdrant (เช่น เรื่อง logistics, retail) แล้วถาม Agent ดู

In [ ]:
# 🧪 แบบฝึกหัด: ทดสอบ RAG Agent
# await demo_chat(rag_agent, [
#     'NLP ภาษาไทย มีปัญหาอะไร',
#     'Qdrant ดียังไง',
# ])

---
## 👥 Section 2.4: Multi-Agent System

### ทำไมต้อง Multi-Agent?

Agent เดียวทำทุกอย่าง → สับสนเมื่อซับซ้อน  
Multi-Agent: **แบ่งงานตามความเชี่ยวชาญ** → แม่นยำกว่า

### 🧩 3 รูปแบบ Workflow ใน ADK

| Pattern | ลักษณะ | เมื่อไหร่ใช้? |
|---------|--------|-------------|
| **Sequential** | ทำทีละขั้น A→B→C | ข้อมูลต้องส่งต่อ เช่น ค้นหา→สรุป→แปล |
| **Parallel** | ทำพร้อมกัน A‖B‖C | งานอิสระ ไม่ต้องรอกัน เช่น ค้น 3 แหล่งพร้อมกัน |
| **Loop** | ทำซ้ำจนกว่าจะพอใจ | ปรับปรุงคำตอบ เช่น เขียน→ตรวจ→แก้→ตรวจ... |

```
Sequential:    [A] → [B] → [C]          ทำตามลำดับ

Parallel:      [A] ─┐
               [B] ─┤─→ รวมผลลัพธ์      ทำพร้อมกัน
               [C] ─┘

Loop:          [A] → [B] →╮              ทำซ้ำจนกว่า
                ╰─────────╯              จะผ่านเกณฑ์
```

### 1️⃣ SequentialAgent — ทำทีละขั้น

```
ค้นหาข้อมูล → สรุปให้กระชับ → แปลเป็นอังกฤษ
   Step 1          Step 2           Step 3
```

แต่ละ agent ส่งผลลัพธ์ต่อไปให้ agent ถัดไปผ่าน **shared state**

In [ ]:
%%time
from google.adk.agents import SequentialAgent

# Sub-agent 1: ค้นหาข้อมูล
step1_search = Agent(
    name='step1_search',
    model='gemini-2.5-flash',
    description='ค้นหาข้อมูล AI ในไทย',
    instruction='ค้นหาข้อมูลจากฐานความรู้แล้วตอบเป็นข้อเท็จจริง',
    tools=[search_thai_ai_knowledge]
)

# Sub-agent 2: สรุป
step2_summary = Agent(
    name='step2_summary',
    model='gemini-2.5-flash',
    description='สรุปข้อมูลให้กระชับ',
    instruction='รับข้อมูลจาก step ก่อนหน้า สรุปเป็น 3 bullet points ภาษาไทย'
)

# Sub-agent 3: แปลภาษา
step3_translate = Agent(
    name='step3_translate',
    model='gemini-2.5-flash',
    description='แปลเป็นภาษาอังกฤษ',
    instruction='แปลข้อมูลที่ได้รับเป็นภาษาอังกฤษ ให้เป็นธรรมชาติ'
)

# SequentialAgent: ค้น → สรุป → แปล
sequential_pipeline = SequentialAgent(
    name='search_summarize_translate',
    description='Pipeline: ค้นหา → สรุป → แปลภาษา',
    sub_agents=[step1_search, step2_summary, step3_translate]
)

print('✅ สร้าง SequentialAgent: ค้น → สรุป → แปล')
print(f'   ลำดับ: {[a.name for a in sequential_pipeline.sub_agents]}')

In [ ]:
runner = InMemoryRunner(agent=sequential_pipeline, app_name=sequential_pipeline.name)
await runner.session_service.create_session(app_name=sequential_pipeline.name, user_id='user_1', session_id='test_session')
# ─── ทดสอบ Sequential Pipeline ───
print('═' * 60)
print('🧪 SequentialAgent: ค้น → สรุป → แปล')
print('═' * 60)

response = await chat_with_agent(runner, 'AI ช่วยเกษตรกรไทยอย่างไร', session_id='test_session')
print(f'\n📋 ผลลัพธ์สุดท้าย (ผ่าน 3 ขั้นตอน):')
print(response)

### 💡 สังเกต
- ผลลัพธ์สุดท้ายเป็น **ภาษาอังกฤษ** เพราะผ่าน step3_translate
- ถ้าสลับลำดับ sub_agents → ผลลัพธ์เปลี่ยน!
- **Shared State**: แต่ละ agent เห็นผลลัพธ์ของ agent ก่อนหน้า

### 2️⃣ ParallelAgent — ทำพร้อมกัน

| Input | Task | Output |
|:-----:|------|:------:|
| คำถาม | → ค้นเรื่อง healthcare | รวมผลลัพธ์ |
| | → ค้นเรื่อง banking | |
| | → ค้นเรื่อง education | |

**เร็วกว่า Sequential** เพราะทำ 3 งานพร้อมกัน แทนที่จะรอทีละอัน

In [ ]:
%%time
from google.adk.agents import ParallelAgent

# 3 agents ค้นหาคนละหัวข้อ พร้อมกัน
healthcare_searcher = Agent(
    name='healthcare_searcher',
    model='gemini-2.5-flash',
    description='ค้นหาเรื่อง AI ด้านสาธารณสุข',
    instruction='ค้นหาข้อมูลเกี่ยวกับ AI ในวงการแพทย์และสาธารณสุขไทย ตอบสั้น 2-3 ประโยค',
    tools=[search_thai_ai_knowledge]
)

banking_searcher = Agent(
    name='banking_searcher',
    model='gemini-2.5-flash',
    description='ค้นหาเรื่อง AI ด้านการเงิน',
    instruction='ค้นหาข้อมูลเกี่ยวกับ AI ในวงการธนาคารและการเงินไทย ตอบสั้น 2-3 ประโยค',
    tools=[search_thai_ai_knowledge]
)

education_searcher = Agent(
    name='education_searcher',
    model='gemini-2.5-flash',
    description='ค้นหาเรื่อง AI ด้านการศึกษา',
    instruction='ค้นหาข้อมูลเกี่ยวกับ AI ในวงการการศึกษาไทย ตอบสั้น 2-3 ประโยค',
    tools=[search_thai_ai_knowledge]
)

# ParallelAgent: ค้น 3 หัวข้อพร้อมกัน!
parallel_search = ParallelAgent(
    name='parallel_multi_topic_search',
    description='ค้นหา 3 หัวข้อพร้อมกัน',
    sub_agents=[healthcare_searcher, banking_searcher, education_searcher]
)

print('✅ สร้าง ParallelAgent: ค้น 3 หัวข้อพร้อมกัน')
print(f'   Agents: {[a.name for a in parallel_search.sub_agents]}')

In [ ]:
runner = InMemoryRunner(agent=parallel_search, app_name=parallel_search.name)
await runner.session_service.create_session(app_name=parallel_search.name, user_id='user_1', session_id='test_session')
# ─── ทดสอบ Parallel Search ───
print('═' * 60)
print('🧪 ParallelAgent: ค้น 3 หัวข้อพร้อมกัน')
print('═' * 60)

response = await chat_with_agent(runner, 'สรุปภาพรวม AI ในประเทศไทย', session_id='test_session')
print(f'\n📋 ผลรวม (ค้นพร้อมกัน 3 ด้าน):')
print(response)

### 💡 สังเกต
- 3 agents ทำงาน **พร้อมกัน** → เร็วกว่า Sequential 3 เท่า (ในทางทฤษฎี)
- แต่ละ agent **อิสระ** ไม่เห็นผลลัพธ์ของคนอื่น
- เหมาะกับงานที่ **ไม่ต้องส่งต่อข้อมูล** ระหว่างกัน

### 3️⃣ LoopAgent — ทำซ้ำจนผ่านเกณฑ์

| Step | Action | Result |
|:----:|--------|--------|
| 1 | เขียนคำตอบ | draft |
| 2 | ตรวจสอบคุณภาพ | ยังไม่ดี? → กลับ Step 1 |
| 3 | ผ่านเกณฑ์ | ✅ จบ |

**ใช้สำหรับ self-improvement**: Agent เขียน → Agent ตรวจ → ถ้ายังไม่ดีก็แก้ไขแล้วตรวจใหม่

> ⚠️ ต้องกำหนด `max_iterations` เพื่อป้องกันวนลูปไม่จบ!

In [ ]:
%%time
from google.adk.agents import LoopAgent

# Agent 1: เขียนสรุป
writer_agent = Agent(
    name='writer_agent',
    model='gemini-2.5-flash',
    description='เขียนสรุปเนื้อหา',
    instruction="""เขียนสรุปเนื้อหาที่ได้รับ ให้กระชับ เข้าใจง่าย
    ถ้ามี feedback จากรอบก่อน ให้ปรับปรุงตาม""",
    tools=[search_thai_ai_knowledge]
)

# Agent 2: ตรวจสอบคุณภาพ (critic)
critic_agent = Agent(
    name='critic_agent',
    model='gemini-2.5-flash',
    description='ตรวจสอบคุณภาพคำตอบ ให้ feedback',
    instruction="""ตรวจสอบคำตอบที่ได้รับ:
    - มีข้อมูลครบถ้วนหรือไม่?
    - เข้าใจง่ายหรือไม่?
    - มีตัวเลขหรือข้อเท็จจริงอ้างอิงหรือไม่?
    
    ถ้ายังไม่ดี → แนะนำสิ่งที่ต้องปรับปรุง
    ถ้าดีแล้ว → ตอบว่า 'APPROVED: [สรุปฉบับสุดท้าย]'
    """
)

# LoopAgent: เขียน → ตรวจ → แก้ → ตรวจ → ... (สูงสุด 3 รอบ)
loop_refiner = LoopAgent(
    name='refine_loop',
    description='ปรับปรุงคำตอบจนกว่าจะผ่านเกณฑ์',
    sub_agents=[writer_agent, critic_agent],
    max_iterations=3  # ป้องกันลูปไม่จบ!
)

print('✅ สร้าง LoopAgent: เขียน ↔ ตรวจ (สูงสุด 3 รอบ)')
print(f'   Max iterations: {loop_refiner.max_iterations}')

In [ ]:
runner = InMemoryRunner(agent=loop_refiner, app_name=loop_refiner.name)
await runner.session_service.create_session(app_name=loop_refiner.name, user_id='user_1', session_id='test_session')
# ─── ทดสอบ Loop Agent ───
print('═' * 60)
print('🧪 LoopAgent: เขียน → ตรวจ → แก้ไข → ตรวจ ...')
print('═' * 60)

response = await chat_with_agent(runner, 'สรุปว่า RAG ทำงานอย่างไร พร้อมตัวอย่างใช้จริงในไทย', session_id='test_session')
print(f'\n📋 ผลลัพธ์สุดท้าย (หลังปรับปรุง):')
print(response)

### 💡 สังเกต
- Writer เขียนสรุป → Critic ตรวจ → ถ้ายังไม่ดี Critic ให้ feedback → Writer แก้ไข
- LoopAgent หยุดเมื่อ: (1) Critic ตอบ APPROVED หรือ (2) ถึง `max_iterations`
- **ยิ่งวนมากรอบ → คำตอบยิ่งดี** แต่ก็ใช้เวลามากขึ้น

> ⚠️ ถ้าไม่ตั้ง `max_iterations` → อาจวนลูปไม่จบ!

### 📊 เปรียบเทียบ 3 รูปแบบ + LLM-based Routing

| รูปแบบ | Class | Flow | ใช้ LLM ควบคุม? | ตัวอย่าง |
|--------|-------|------|---------------|----------|
| **Sequential** | `SequentialAgent` | A→B→C | ❌ Deterministic | ค้น→สรุป→แปล |
| **Parallel** | `ParallelAgent` | A‖B‖C | ❌ Deterministic | ค้น 3 แหล่งพร้อมกัน |
| **Loop** | `LoopAgent` | A↔B (ซ้ำ) | ❌ Deterministic | เขียน↔ตรวจ จนดี |
| **LLM Routing** | `Agent` + `sub_agents` | LLM เลือก | ✅ LLM ตัดสินใจ | Orchestrator ส่งงาน |

> 💡 **Workflow Agents** (Sequential/Parallel/Loop) ทำงาน **deterministic** = ได้ผลเหมือนเดิมทุกครั้ง  
> ส่วน **LLM Routing** (Agent + sub_agents) ให้ LLM ตัดสินใจเอง = ยืดหยุ่นกว่าแต่ผลอาจต่างกัน

### 4️⃣ LLM-based Routing — Agent ตัดสินใจส่งงาน

ใช้ `Agent` + `sub_agents` → **LLM เลือกเอง** ว่าจะส่งงานให้ใคร

In [ ]:
%%time
# ─── LLM-based Routing: Orchestrator ───

# Sub-agents เฉพาะทาง
search_agent = Agent(
    name='search_agent',
    model='gemini-2.5-flash',
    description='ค้นหาข้อมูลจากฐานความรู้ AI ไทย',
    instruction='ค้นหาข้อมูล ตอบแค่ข้อเท็จจริงที่พบ',
    tools=[search_thai_ai_knowledge]
)

summarizer_agent = Agent(
    name='summarizer_agent',
    model='gemini-2.5-flash',
    description='สรุปข้อมูลให้กระชับ เข้าใจง่าย',
    instruction='สรุปเป็น bullet points 3-5 ข้อ ใส่ emoji'
)

translator_agent = Agent(
    name='translator_agent',
    model='gemini-2.5-flash',
    description='แปลข้อความระหว่างภาษาไทยและอังกฤษ',
    instruction='แปลภาษาให้เป็นธรรมชาติ ไม่แปลตรงตัว'
)

# Orchestrator: LLM เลือกเอง
orchestrator = Agent(
    name='thai_ai_orchestrator',
    model='gemini-2.5-flash',
    description='Agent หลักที่ดูแลทีม',
    instruction="""คุณเป็น Agent หลัก มีทีม:
    1. search_agent — ค้นหาข้อมูล
    2. summarizer_agent — สรุป
    3. translator_agent — แปลภาษา
    ส่งงานให้ทีมตามความเหมาะสม""",
    sub_agents=[search_agent, summarizer_agent, translator_agent]
)

print('✅ สร้าง LLM Routing Orchestrator')

In [ ]:
# ─── ทดสอบ LLM Routing ───
print('═' * 60)
print('🧪 LLM Routing: Agent เลือกส่งงานให้ทีม')
print('═' * 60)

await demo_chat(orchestrator, [
    'AI ในวงการเกษตรไทยเป็นอย่างไร',
    'ช่วยสรุปให้หน่อย สั้นๆ',
    'แปลเป็นอังกฤษ',
])

### 💡 สังเกต
- Orchestrator **ไม่ได้โค้ดกำหนด** ว่าส่งงานให้ใคร → **LLM เลือกเอง** จาก description
- ถาม "ค้นหา..." → ส่งให้ 
- ถาม "สรุป..." → ส่งให้ 
- ถาม "แปล..." → ส่งให้ 

> ⚡ **ข้อดี**: ยืดหยุ่นมาก เพิ่ม agent ใหม่ได้ง่าย  
> ⚠️ **ข้อเสีย**: LLM อาจส่งงานผิดคน ผลอาจต่างกันทุกครั้ง

### 🧪 แบบฝึกหัด 2.4

1. ลองสร้าง `SequentialAgent` ใหม่: ค้น → วิเคราะห์ → สร้าง quiz
2. ลองใช้ `ParallelAgent` เทียบ AI ในไทย vs อังกฤษ vs ญี่ปุ่น
3. ลอง `LoopAgent` ให้ agent แต่งกลอนแล้วตรวจ จนได้กลอนที่ดี

In [ ]:
# 🧪 แบบฝึกหัด: เลือก pattern ที่ชอบแล้วลองสร้างเอง
# Hint: SequentialAgent, ParallelAgent, LoopAgent
# from google.adk.agents import SequentialAgent  # etc.


---
## 🧠 Section 2.5: Session & Memory

### Agent จำได้ — ด้วย Session

```
ไม่มี Session:     ทุกข้อความเริ่มต้นใหม่ (ลืมหมด)
มี Session:        จำบทสนทนาที่ผ่านมา → ตอบต่อเนื่องได้
```

### Session ประกอบด้วยอะไรบ้าง?

| ส่วนประกอบ | คืออะไร | ตัวอย่าง |
|-----------|--------|--------|
| **History** | ประวัติข้อความ user ↔ agent | "ถาม: AI คืออะไร" → "ตอบ: AI คือ..." |
| **State** | ข้อมูลที่เก็บไว้ระหว่าง turn | ชื่อผู้ใช้, preferences, ผลลัพธ์ tool |
| **user_id** | ระบุตัวตนผู้ใช้ | "student_001" |
| **session_id** | ระบุบทสนทนา | "session_abc123" |

### SessionService แบบไหนบ้าง?

| ชนิด | เก็บที่ | ปิดโปรแกรม | ใช้เมื่อ |
|------|--------|-----------|--------|
| `InMemorySessionService` | RAM | ❌ หายหมด | Dev/ทดสอบ |
| `DatabaseSessionService` | DB | ✅ ยังอยู่ | Production |

> 💡 ใน workshop เราใช้ `InMemorySessionService` เพราะง่ายและเร็ว

In [ ]:
# ─── Demo: Agent จำบริบทได้ ───
async def conversation_with_memory(agent, messages):
    """สนทนาหลายรอบ ใช้ session เดียวกัน → Agent จำได้"""
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    
    await runner.session_service.create_session(
        app_name=agent.name,
        user_id='student_demo',
        session_id='memory_demo'
    )

    for msg in messages:
        print(f'\n👤 ผู้ใช้: {msg}')
        content = types.Content(
            role='user',
            parts=[types.Part.from_text(text=msg)]
        )

        final_response = ''
        async for event in runner.run_async(
            user_id='student_demo',
            session_id='memory_demo',
            new_message=content
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        final_response = part.text
        
        print(f'🤖 Agent: {final_response}')

# ทดสอบ
await conversation_with_memory(greeting_agent, [
    'สวัสดีครับ ผมชื่อสมชาย',
    'ผมสนใจเรื่อง AI ในวงการสุขภาพ',
    'แล้วในวงการการเงินล่ะ',
    'เปรียบเทียบสองวงการนี้ให้หน่อย',
    'จำได้ไหมว่าผมชื่ออะไร?',
])

### 💡 สังเกต
- ถาม "แล้วในวงการการเงินล่ะ" → Agent **จำได้** ว่ากำลังคุยเรื่อง AI ในไทย
- ถาม "เปรียบเทียบสองอันนี้" → Agent **จำได้** ว่าคุยเรื่อง healthcare + banking
- ถ้าใช้ `chat_with_agent` (session ใหม่ทุกครั้ง) → Agent จะลืมหมด

> **Session = ความจำ**: ใช้ session เดียวกัน → Agent จำบริบทได้

---
## 🚀 Section 2.6: Agentic RAG เต็มรูปแบบ ⭐

รวมทุกอย่าง: **Agent + RAG Tool + Multi-Agent + Memory**

| Component | Role |
|---|---|
| 🎯 **Study Assistant** | Orchestrator Agent |
| 🔍 Search Agent (Qdrant) | ค้นหาข้อมูล |
| 📝 Summarizer Agent | สรุปข้อมูล |
| 🌐 Translator Agent | แปลภาษา |
| 💾 Session | จำบทสนทนา |

In [ ]:
# ─── สร้าง Study Assistant เต็มรูปแบบ ───

study_assistant = Agent(
    name='study_assistant',
    model='gemini-2.5-flash',
    description='ผู้ช่วยเรียน AI/RAG สำหรับนักศึกษาไทย',
    instruction="""คุณเป็นผู้ช่วยเรียนวิชา AI/RAG สำหรับนักศึกษาไทย

    ความสามารถ:
    1. search_agent — ค้นหาข้อมูลจากฐานความรู้ (ใช้เมื่อถามเรื่อง AI/RAG ในไทย)
    2. summarizer_agent — สรุปข้อมูลให้เข้าใจง่าย
    3. translator_agent — แปลภาษา

    วิธีทำงาน:
    - เมื่อถูกถามเรื่อง AI → ค้นหาจาก search_agent ก่อน → แล้วสรุปให้เข้าใจง่าย
    - จำบริบทการสนทนา ถ้าผู้ใช้ถาม "แล้ว...ล่ะ" ให้อ้างอิงบริบทก่อนหน้า
    - ตอบเป็นภาษาไทย ใส่ emoji
    - ท้ายคำตอบ เพิ่ม 💡 แนะนำสิ่งที่น่าสนใจเพิ่มเติม
    """,
    sub_agents=[
        Agent(
            name='study_search_agent',
            model='gemini-2.5-flash',
            description='ค้นหาข้อมูลจากฐานความรู้ AI ไทย',
            instruction='ค้นหาข้อมูล ตอบแค่ข้อเท็จจริงที่พบ',
            tools=[search_thai_ai_knowledge]
        ),
        Agent(
            name='study_summarizer_agent',
            model='gemini-2.5-flash',
            description='สรุปข้อมูลให้กระชับ เข้าใจง่าย',
            instruction='สรุปเป็น bullet points 3-5 ข้อ ใส่ emoji'
        ),
        Agent(
            name='study_translator_agent',
            model='gemini-2.5-flash',
            description='แปลข้อความระหว่างไทย-อังกฤษ',
            instruction='แปลให้ถูกต้องตามบริบท'
        ),
    ]
)

print('✅ สร้าง Study Assistant เต็มรูปแบบ!')

In [ ]:
# ─── Demo: Agentic RAG เต็มรูปแบบ ───
print('═' * 60)
print('🎓 Demo: Study Assistant (Agentic RAG เต็มรูปแบบ)')
print('═' * 60)

await conversation_with_memory(study_assistant, [
    'ช่วยอธิบายว่า RAG ทำงานยังไง',
    'มีตัวอย่างใช้จริงในไทยมั้ย',
    'สรุปทั้งหมดให้หน่อย',
    'แปลสรุปเป็นภาษาอังกฤษ',
])

### 🧪 แบบฝึกหัด 2.6

1. ลองถาม Study Assistant ด้วยคำถามของตัวเอง
2. สังเกตว่า Agent ส่งงานไปให้ sub-agent ไหน
3. ลองเพิ่ม tool ใหม่ เช่น quiz_generator

In [ ]:
# 🧪 แบบฝึกหัด: คุยกับ Study Assistant
# await conversation_with_memory(study_assistant, [
#     'embedding ภาษาไทย ใช้ model อะไร',
#     'ทำไมต้อง multilingual',
#     'สรุปสั้นๆ ให้หน่อย',
# ])

---
## 🎯 สรุป: สิ่งที่เราเรียนรู้ในวันนี้

| Section | สิ่งที่เรียน | เครื่องมือ |
|---------|------------|----------|
| 2.1 | Agent = LLM + Instructions | `google.adk.agents.Agent` |
| 2.2 | Tool = Python function ที่ Agent เรียกใช้ | `tools=[function]` |
| 2.3 | RAG Tool = ค้นหา VectorDB | Qdrant + Embedding |
| 2.4 | Multi-Agent = หลาย Agent ทำงานร่วมกัน | `sub_agents=[]` |
| 2.5 | Session = Agent จำบริบท | `InMemorySessionService` |
| 2.6 | Agentic RAG = ระบบเต็มรูปแบบ | ทุกอย่างรวมกัน |

### 🛣️ Day 3: Evaluation
- วัดคุณภาพ RAG ด้วย RAGAS
- Automated Testing
- ปรับปรุง Agent ให้ดีขึ้น

---

**🙏 ขอบคุณที่เรียนด้วยกัน!**